<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
DeepAgents - Meeting/Briefing-Skill
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Dieses Notebook zeigt einen **Kern-Baustein des Leitprojekts**: Meeting-Vorbereitung, Aufgabenverfolgung und Statusreporting des PMO-Assistenten. Der Agent nutzt genau den hier gebauten Skill, um aus Protokollen, Risikolisten und offenen Punkten ein geprüftes Briefing, eine Aufgabenübersicht oder eine Statusübersicht zu erzeugen — mit Structured Output, Freigabe und Quellenbindung wie im gesamten Kurs. Der Skill ersetzt kein vollständiges PMO-System und trifft keine verbindlichen Projektentscheidungen. Was frühere Kursversionen als Transferbeispiel zu einem RAG-zentrierten PMO-Assistenten zeigten, ist hier der direkte Weg zum Leitprojekt.

**Praxis-Transfer:** Mara Vogt erhält aus Meeting-Unterlagen ein strukturiertes Briefing mit Aktionspunkten, Quellenbezug und einem sichtbaren Prüf-Gate.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
    copy_from_github,
)

import os
os.environ["LANGSMITH_TRACING"] = "false"
from langsmith import tracing_context
# Tracing bleibt standardmäßig aus; ausgewählte Referenzläufe werden explizit aktiviert.
os.environ["LANGSMITH_PROJECT"]  = "pmo-briefing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["deepagents"],
    "metadata": {"notebook": "deepagents_skill_pmo_assistent", "version": "1.0"}
}


In [ ]:
#@title 📦 Installationen { display-mode: "form" }

!uv pip install --system -q "deepagents==0.6.1"
!uv pip install --system -q markitdown[all]

# 1 | Übersicht

---

<div style="background-color:#1a1a2e; padding:30px; border-radius:12px; margin-bottom:10px">
  <h1 style="color:#e0e0e0; font-family:monospace; margin:0">PMO-Briefing-Skill mit DeepAgents</h1>
  <p style="color:#a0a0c0; font-family:monospace; margin:8px 0 0 0">Skill-Dateien aus GitHub · Gate-Analyse mit gpt-5.6-sol · structured_output · Writer Sub-Agent</p>
</div>

**Was dieses Notebook zeigt:**
- Skill-Dateien (`SKILL.md`, `WRITER.md`, Regelwerke) **direkt aus dem GitHub-Repo laden**
- Gate-Analyse mit `gpt-5.6-sol` + `with_structured_output()` → deterministisches JSON
- `extract_actions.py` aus GitHub holen, lokal zwischenspeichern, deterministisch aufrufen
- **Writer als Sub-Agent** mit geladenem `WRITER.md` als System-Prompt
- DeepAgent-Koordinator orchestriert Skill-Abarbeitung autonom

---

<font color="black" size="5">Was dieses Notebook zeigt</font>

In `06_skill/pmo-briefing` im GitHub-Repo liegt ein vollständiger Skill:
Kontext laden, Agenda nach festen Regeln strukturieren, Action Items deterministisch
extrahieren und das Ergebnis im definierten Briefing-Format ausgeben.

Dieses Notebook baut daraus einen **DeepAgent** mit vier Kernmerkmalen:

| # | Merkmal | Warum |
|---|---------|-------|
| 1 | **Skill-Dateien aus GitHub laden** | Änderungen am Skill greifen ohne Notebook-Anpassung |
| 2 | **`with_structured_output()`** | Gate-Analyse gibt Pydantic-validiertes JSON zurück |
| 3 | **Modell-Auswahl laut Guide** | `gpt-5.6-sol` für Koordinator & Gate-Analyse |
| 4 | **Writer als Sub-Agent** | Trennung von Analyse und Formatierung |

> **Best Practice:** DeepAgents ist hier die Harness-Schicht, nicht die Wissensquelle.
> Die Fachlogik bleibt im Skill-Repo, in den Regelwerken und im deterministischen Tooling verankert.

In [ ]:
#@markdown   <p><font size="4" color="green">Skill-Architektur</font></p>

diagram_arch = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) --> LOAD
    U([" 🧑 Meeting-Anfrage"]) --> K

    subgraph K_BLOCK [" 🤖 DeepAgent Koordinator — gpt-5.6-sol"]
        LOAD["Skill-Dateien laden\n(copy_from_github)"] --> K
        K["Kontext laden\nAction Items extrahieren"]
        K --> G["🛠️ analysiere_meeting_kontext\ngpt-5.6-sol + with_structured_output"]
        K --> E["🛠️ aktionspunkte_extrahieren\nextract_actions.py als Subprocess"]
        G --> JSON["GateOutputResult JSON\nPydantic-validiert"]
        E --> JSON
        JSON --> W["Sub-Agent\nbriefing-writer"]
    end

    W --> B([" ✅ PMO-Briefing"])

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class GH,U input;
    class B output;
    class LOAD,JSON target;
'''
mermaid(diagram_arch, width=580)

# 2 | Skill-Dateien aus GitHub laden

---



Alle Skill-Dateien werden **einmalig aus dem GitHub-Repo** in ein lokales Cache-Verzeichnis geladen:

| Datei | Zweck |
|-------|-------|
| `SKILL.md` | Gate-Regeln, Hard Rules, Workflow-Definition |
| `references/writer-format.md` | Anweisungen für den Writer Sub-Agenten |
| `references/agenda_rules.md` | Prioritäten, Agenda-Aufbau |
| `references/action_rules.md` | Action-Item-Erkennungsregeln |
| `references/examples.md` | Referenzbeispiele für Gate und Writer |
| `scripts/extract_actions.py` | Deterministisches Skript — direkt im Cache-Verzeichnis |

> 💡 Das Skript `extract_actions.py` liegt nach dem Download direkt im Cache-Verzeichnis
> und wird per `subprocess` aufgerufen — kein Temp-Dateien-Schreiben nötig.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path
from textwrap import dedent
from typing import List, Optional

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from markitdown import MarkItDown
from pydantic import BaseModel, Field
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver


**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

In [ ]:
# 2.1 Skill-Dateien aus GitHub laden und lokal cachen

SKILL_DIR = Path("./_skill_pmo_briefing")

copy_from_github(
    source="ralf-42/Agenten/06_skill/pmo-briefing",
    target=str(SKILL_DIR),
)

SCRIPT_PATH = SKILL_DIR / "scripts" / "extract_actions.py"

_file_map = {
    "skill":        SKILL_DIR / "SKILL.md",
    "writer":       SKILL_DIR / "references" / "writer-format.md",
    "agenda_rules": SKILL_DIR / "references" / "agenda_rules.md",
    "action_rules": SKILL_DIR / "references" / "action_rules.md",
    "examples":     SKILL_DIR / "references" / "examples.md",
}

def read_skill(name: str) -> str:
    """Liest eine Skill-Datei aus dem lokalen Cache."""
    if name not in _file_map:
        return f"Unbekannte Datei: {name}. Verfügbar: {list(_file_map.keys())}"
    return _file_map[name].read_text(encoding="utf-8").strip()

mprint(f"Skill-Verzeichnis: `{SKILL_DIR}`")
mprint(f"Skill-Dateien: {list(_file_map.keys())}")
mprint(f"Skript: `{SCRIPT_PATH}`")

In [ ]:
# 2.2 Demo-Kontext-Dokumente aus GitHub laden und lokal cachen

DATA_DIR = Path("./_data_pmo_briefing")
KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text"
KORPUS_MASKE = "0*.docx"
KORPUS_TARGET = str(DATA_DIR)

copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

_md = MarkItDown()

DEMO_CONTEXTS_VORB = {
    p.name: _md.convert(str(p)).text_content
    for p in sorted(DATA_DIR.glob("0*.docx"))
}

# Nachbereitung: Steuerkreis vom 28.04.2026 (inline, Auszug aus dem Protokoll)
DEMO_CONTEXTS_NACH = {
    "steuerkreis_2026-04-28.md": dedent("""
        Protokollauszug — Steuerkreis Projekt Kompass, 28.04.2026
        Teilnehmer: Mara Vogt, Dr. Thomas Brandt, Jonas Reiter.

        Entschieden: Der Status von Risiko R2 (Datenschutzfreigabe) wird von offen auf mitigiert geändert (D3).
        Die Freigabe von Dr. Thomas Brandt gilt für den kuratierten Testkorpus; für produktive Daten ist eine separate Prüfung erforderlich.

        Nächste Schritte: Mara Vogt aktualisiert die Risikoliste und vermerkt den Statuswechsel.
        Dr. Thomas Brandt liefert Formulierungen für Datenschutzgrenzen.
        Jonas Reiter soll technische Logging- und Zugriffskonzepte vorbereiten.

        Offener Punkt: Mitigiert bedeutet nicht erledigt; das Risiko bleibt beobachtet.
    """)
}

mprint(f"Vorkontext-Dokumente geladen: `{len(DEMO_CONTEXTS_VORB)}`")


In [ ]:
# 2.3 Wissensdatenbank (ChromaDB) aufbauen — ergänzt die Dokument-Tools, ersetzt sie nicht
import shutil
from genai_lib.pmo_rag import get_pmo_vectorstore, index_pmo_corpus, make_suche_wissensdatenbank_tool

KB_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_pmo_assistent"
KB_DIR = Path("./_data_pmo_korpus")
KB_CHROMA_DIR = "./chroma_skill_pmo"   # lokal, ohne Drive-Mount (auf Colab nur temporär)

shutil.rmtree(KB_DIR, ignore_errors=True)
copy_from_github(source=KB_QUELLE, target=str(KB_DIR), mask="*")

vectorstore = get_pmo_vectorstore(persist_directory=KB_CHROMA_DIR, collection_name="pmo_korpus_skill_pmo")
index_pmo_corpus(vectorstore, str(KB_DIR), chunk_size=300, chunk_overlap=30)

mprint(f"Wissensdatenbank: `{vectorstore._collection.count()}` Chunks in `{KB_CHROMA_DIR}`")


In [ ]:
from genai_lib.utilities import load_prompt

# Auszüge aus den lokalen Skill-Dateien (Frontmatter wird automatisch entfernt)
_previews = {
    "SKILL.md":        (str(SKILL_DIR / "SKILL.md"),                        600),
    "agenda_rules.md": (str(SKILL_DIR / "references" / "agenda_rules.md"),  400),
    "WRITER.md":       (str(SKILL_DIR / "references" / "writer-format.md"), 300),
}

zeilen = []
for name, (path, limit) in _previews.items():
    text = load_prompt(path, mode="S")[:limit]
    zeilen += [f"### {name} (Auszug)", "", text, ""]

mprint("\n".join(zeilen))

# 3 | Pydantic-Modelle & Gate-Analyse

---



Damit der Gate-Agent **deterministisch** strukturierten Output liefert,
verwenden wir `with_structured_output()` mit Pydantic-Modellen.

| Modell | Aufgaben |
|--------|---------|
| `ActionItemResult` | Einzelnes Action Item (Was / Wer / Wann) |
| `AgendaPointResult` | Agenda-Punkt (Priorität, Ziel, Dauer, Eigentümer) |
| `GateOutputResult` | Vollständiger strukturierter Gate-Output für den Writer |

> 💡 **Warum `with_structured_output()`?**
> Der Writer-Sub-Agent braucht verlässliche Eingabedaten. Ein freier LLM-Text-Output
> kann in Struktur und Vollständigkeit variieren — ein Pydantic-validiertes JSON nicht.

In [ ]:
# 3.1 Pydantic-Modelle für strukturierten Gate-Output

class ActionItemResult(BaseModel):
    """Ein konkretes Action Item aus dem Meeting."""
    task:  str = Field(description="Aufgabenbeschreibung — kurz, aktionsorientiert")
    owner: str = Field(description="Verantwortliche Person — '[offen]' wenn unklar")
    due:   str = Field(description="Fälligkeitsdatum — '[offen]' wenn unbekannt")


class AgendaPointResult(BaseModel):
    """Ein Agenda-Punkt mit Priorität, Ziel und Zeitplan."""
    priority: str = Field(description="Priorität: hoch, mittel oder gering")
    point:    str = Field(description="Bezeichnung des Agenda-Punkts")
    goal:     str = Field(description="Ziel: entscheiden, klären, informieren oder präsentieren")
    duration: int = Field(description="Geplante Dauer in Minuten")
    owner:    str = Field(description="Verantwortliche Person für diesen Punkt")


class GateOutputResult(BaseModel):
    """Strukturierter Gate-Output — Eingabe für den Writer-Sub-Agenten."""
    type:           str                    = Field(description="vorbereitung oder nachbereitung")
    meeting_type:   str                    = Field(description="Projektmeeting, Kundengespräch, Review, Workshop oder Sonstiges")
    topic:          str                    = Field(description="Meeting-Thema")
    date:           Optional[str]          = Field(default=None, description="Datum/Uhrzeit wenn bekannt")
    participants:   List[str]              = Field(description="Teilnehmerliste")
    agenda:         List[AgendaPointResult]      = Field(description="Agenda-Punkte sortiert: hoch → mittel → gering")
    open_questions: List[str]              = Field(description="Offene Fragen vor dem Meeting")
    action_items:   List[ActionItemResult] = Field(description="Action Items aus Kontext und Vorgesprächen")
    risks:          List[str]              = Field(description="Bekannte Risiken und Konflikte")
    status:         str                    = Field(description="ok, no_context oder conflict")
    conflict:       Optional[bool]         = Field(default=False, description="True bei widersprüchlichen Quellinformationen")
    decisions:      Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: getroffene Entscheidungen")
    open_points:    Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: vertagte Punkte")


# Gate-Constraint: gpt-5.6-sol explizit auf Datenextraktion beschränken
_GATE_CONSTRAINT = """
## PFLICHT: Ausgabeformat

Du bist ein Datenextraktor — kein Schreiber.

- **Gib AUSSCHLIESSLICH die strukturierten Felder aus GateOutputResult zurück.**
- Formuliere KEIN PMO-Briefing, keinen Report, keinen Fließtext.
- Schreibe KEINE Tabellen, KEINE Markdown-Sections, KEINEN Abschlusssatz.
- Diese Regel überschreibt alle anderen Anweisungen.
"""

# Gate-Prompt aus GitHub-Skill-Dateien zusammensetzen
_GATE_RULES = (
    read_skill("skill") + "\n\n"
    + read_skill("agenda_rules") + "\n\n"
    + read_skill("action_rules") + "\n\n"
    + _GATE_CONSTRAINT
)

gate_prompt = ChatPromptTemplate([
    ("system", _GATE_RULES),
    ("user", "Meeting-Anfrage: {anfrage}\n\nKontext-Dokumente:\n{kontext}"),
])

mprint("✅ **Pydantic-Modelle** definiert: `ActionItemResult`, `AgendaPointResult`, `GateOutputResult`")
mprint("✅ **Gate-Prompt** aus GitHub: `SKILL.md` + `agenda_rules.md` + `action_rules.md`")

In [ ]:
# 3.2 Gate-Analyse als @tool — gpt-5.6-sol + with_structured_output()

@tool
def analysiere_meeting_kontext(anfrage: str, kontext: str) -> str:
    """
    Analysiert Meeting-Kontext mit gpt-5.6-sol und gibt strukturierten Gate-Output zurück.

    Verwendet with_structured_output(GateOutputResult) für deterministisches,
    Pydantic-validiertes JSON. Regeln kommen aus den GitHub-Skill-Dateien.

    Args:
        anfrage: Meeting-Typ, Thema, Teilnehmer, Datum
        kontext: Gesamter Meeting-Kontext aus allen geladenen Dokumenten

    Returns:
        Pydantic-validiertes JSON (GateOutputResult) als String
    """
    gate_llm   = init_chat_model(JUDGE)   # kein temperature!
    gate_chain = gate_prompt | gate_llm.with_structured_output(GateOutputResult)
    result: GateOutputResult = gate_chain.invoke({"anfrage": anfrage, "kontext": kontext}, config={**run_cfg, "run_name": "kap3_analysiere_meeting_kontext"})
    return result.model_dump_json(indent=2)


mprint("✅ **`analysiere_meeting_kontext`** — gpt-5.6-sol + `with_structured_output(GateOutputResult)`")

**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

# 4 | Skill-Tools kapseln

---



Der Koordinator bekommt sieben fokussierte Tools:

| Tool | Quelle | Aufgaben |
|------|--------|---------|
| `list_context_documents` | Notebook | Verfügbare Demo-Dokumente anzeigen |
| `load_context_document` | Notebook | Einzelnes Kontextdokument laden |
| `suche_wissensdatenbank` | **ChromaDB** | Semantische Suche im Projektkorpus (ergänzend zu den Dokument-Tools) |
| `load_skill_asset` | **GitHub** | Skill-Regelwerk zur Laufzeit nachladen |
| `aktionspunkte_extrahieren` | **GitHub** (Temp) | Deterministisch via `extract_actions.py` |
| `analysiere_meeting_kontext` | LLM-Chain | gpt-5.6-sol + `with_structured_output(GateOutputResult)` |
| `get_demo_meeting_request` | Notebook | Demo-Anfrage für Beispiel 1 |

In [ ]:
# 4.1 Tool-Definitionen

@tool
def list_context_documents(typ: str = "vorbereitung") -> str:
    """Listet verfügbare Demo-Kontextdokumente für den angegebenen Meeting-Typ.

    Args:
        typ: 'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    return json.dumps(sorted(docs.keys()), ensure_ascii=False)


@tool
def load_context_document(name: str, typ: str = "vorbereitung") -> str:
    """Lädt ein einzelnes Demo-Kontextdokument.

    Args:
        name: Dateiname (aus list_context_documents)
        typ:  'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    if name not in docs:
        return f"Dokument '{name}' nicht gefunden. Verfügbar: {sorted(docs)}"
    return docs[name]


@tool
def load_skill_asset(name: str) -> str:
    """Lädt eine Skill-Datei aus dem GitHub-Repo (SKILL.md, WRITER.md, Regelwerke, Beispiele).

    Erlaubte Werte: skill, writer, agenda_rules, action_rules, examples

    Quelle: github.com/ralf-42/Agenten — 06_skill/pmo-briefing/
    """
    key = name.strip().lower()
    if key not in _file_map:
        return f"Unbekanntes Asset '{name}'. Verfügbar: {list(_file_map.keys())}"
    return read_skill(key)   # aus Cache oder frisch von GitHub


@tool
def aktionspunkte_extrahieren(text: str) -> str:
    """Extrahiert Action Items deterministisch über extract_actions.py (aus GitHub, lokal gecacht).

    Erkennt Aktionsverben, Eigentümer und Fälligkeitsdaten.
    Gibt JSON mit top 5 priorisierten Action Items + Backlog zurück.
    """
    payload = json.dumps({"text": text}, ensure_ascii=False)
    proc = subprocess.run(
        [sys.executable, str(SCRIPT_PATH), payload],
        capture_output=True, text=True, encoding="utf-8", check=False,
    )
    if proc.returncode != 0:
        return json.dumps({
            "status": "error",
            "stderr": proc.stderr.strip(),
            "stdout": proc.stdout.strip(),
        }, ensure_ascii=False, indent=2)
    return proc.stdout.strip()


@tool
def get_demo_meeting_request() -> str:
    """Liefert die Demo-Meeting-Anfrage für Beispiel 1 (Vorbereitung des nächsten Steuerkreises)."""
    return dedent("""
        Thema: Nächster Steuerkreis Projekt Kompass
        Typ: Vorbereitung
        Datum/Zeit: [offen]
        Teilnehmer: Mara Vogt (Projektleitung), Jonas Reiter (IT-Architektur), Dr. Thomas Brandt (Datenschutz/Compliance), Elif Kaya (Fachbereich Vertrieb), Sabine Holler (Sponsorin)
        Dauer: [offen]
        Ziel: PMO-Briefing zur vertagten Entscheidung über die Rechte- und Rollensteuerung vorbereiten.
    """).strip()


# Semantische Suche im Projektkorpus (ChromaDB aus Abschnitt 2.3) — ergänzt die Dokument-Tools
suche_wissensdatenbank = make_suche_wissensdatenbank_tool(vectorstore)

ALLE_TOOLS = [
    get_demo_meeting_request,
    list_context_documents,
    load_context_document,
    suche_wissensdatenbank,
    load_skill_asset,
    aktionspunkte_extrahieren,
    analysiere_meeting_kontext,
]

mprint(f"✅ **{len(ALLE_TOOLS)} Tools** registriert: {[t.name for t in ALLE_TOOLS]}")

In [ ]:
# 4.2 aktionspunkte_extrahieren — Smoke-Test
import pandas as pd
kombinierter_kontext = "\n\n".join(
    [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_VORB.items()]
)

raw = aktionspunkte_extrahieren.invoke({"text": kombinierter_kontext}, config={**run_cfg, "run_name": "kap4_aktionspunkte_extrahieren_raw"})
result = json.loads(raw)

mprint("### 🔍 Extrahierte Action Items (Vorbereitung-Kontext)")
df_items = pd.DataFrame(result.get("action_items", []), columns=["task", "owner", "due"])
df_items.columns = ["Aufgaben", "Verantwortlich", "Fällig"]
df_items.insert(0, "#", range(1, len(df_items) + 1))
display(df_items.style.hide(axis="index"))
if result.get("backlog_count", 0) > 0:
    mprint(f'> ⚠️ **Backlog:** {result["backlog_count"]} weitere Item(s)')


<font color="black" size="5">Best-Practice-Hinweise für dieses Beispiel</font>

- **Version pinnen:** `deepagents==0.6.1` — stabile Basis für `subagents=[...]`-Syntax und HarnessProfile.
- **Skill-Dateien aus GitHub:** Regeländerungen in `SKILL.md` wirken beim nächsten Notebook-Run
  automatisch — kein manuelles Copy-Paste in Prompts.
- **`references/writer-format.md` laden:** `load_prompt(url, mode="S")` entfernt automatisch ein etwaiges Frontmatter.
  Kein notebook-spezifisches `read_skill()` nötig.
- **Skript als Temp-Datei:** `extract_actions.py` wird beim Setup in `tempfile.gettempdir()`
  geschrieben. Kein permanentes File-Schreiben — sauber für Cloud-Umgebungen.
- **`recursion_limit` großzügig setzen:** Planning, Tool-Calls und Sub-Agent-Delegation
  zählen alle als eigene Schritte.
- **Modell pro Sub-Agent:** Das `model`-Feld im Sub-Agent-Dict wird unterstützt.
  Writer läuft auf `gpt-5.4-mini` (WORKER), Koordinator auf `o3-mini` (ROUTER) — explizit im Code.

# 5 | DeepAgent aufbauen

---

In [ ]:
# 5.1 Writer Sub-Agent — System-Prompt via load_prompt() aus lokalem Cache (references/writer-format.md)
writer_subagent = {
    "name": "briefing-writer",
    "description": "Formatiert den Gate-Output exakt im Ausgabeformat des PMO-Briefing-Skills.",
    "model": init_chat_model(WORKER),   # gpt-5.4-mini — Textqualität für Briefings
    "system_prompt": (
        load_prompt(str(SKILL_DIR / "references" / "writer-format.md"), mode="S")   # lokal aus Cache
        + "\n\n"
        + "Gib ausschließlich das fertige Briefing aus.\n"
        + "Keine Websuche, keine neuen Fakten, keine Spekulation."
    ),
    "tools": [],
}

# 5.2 DeepAgent Koordinator
KOORDINATOR_PROMPT = (
    "Du bist der Koordinator für den technischen Skill `pmo-briefing`.\n\n"
    "Pflicht-Ablauf:\n"
    "1. get_demo_meeting_request aufrufen (Beispiel 1) ODER Anfrage aus Nachricht lesen\n"
    "2. list_context_documents aufrufen und ALLE Dokumente mit load_context_document laden\n"
    "   Optional: Für frühere Entscheidungen, Risiken oder offene Punkte suche_wissensdatenbank nutzen (ergänzt das Laden, ersetzt es nicht)
"
    "3. aktionspunkte_extrahieren für den gesamten geladenen Kontext aufrufen\n"
    "4. analysiere_meeting_kontext aufrufen → GateOutputResult JSON erhalten\n"
    "5. GateOutputResult an briefing-writer delegieren zur finalen Formatierung\n\n"
    "Hard Rules:\n"
    "- IMMER aktionspunkte_extrahieren aufrufen — nie Action Items frei erfinden\n"
    "- IMMER analysiere_meeting_kontext aufrufen — kein freies Briefing-Schreiben\n"
    "- Treffer aus suche_wissensdatenbank nur mit Quellenangabe verwenden; die geladenen Dokumente haben Vorrang
"
    "- Fehlende Angaben als [offen] markieren, nie spekulieren\n"
    "- Antwort auf Deutsch"
)

meeting_agent = create_deep_agent(
    model=init_chat_model(ROUTER, reasoning_effort="none"),   # Koordinator → gpt-5.6-luna (Tools → reasoning_effort="none")
    tools=ALLE_TOOLS,
    subagents=[writer_subagent],
    system_prompt=KOORDINATOR_PROMPT,
    checkpointer=InMemorySaver(),
)

mprint("✅ **DeepAgent** erstellt")
mprint(f"   Koordinator: `gpt-5.6-luna`  ·  Sub-Agent: `briefing-writer`  ·  Tools: {len(ALLE_TOOLS)}")

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

<p><font color='black' size="5">
5.3 HarnessProfile — Modell-spezifische Optimierung
</font></p>

DeepAgents liefert **eingebaute Profile** für OpenAI und Anthropic.
Sie aktivieren automatisch und verbessern die Agent-Qualität laut Benchmark um 10–20 Punkte — ohne Code-Änderungen.

**Wann ein eigenes `HarnessProfile` sinnvoll ist:**

| Anwendungsfall | Beispiel |
|---------------|---------|
| Sprachvorgabe | Deutsch erzwingen, auch wenn Skill-Dateien auf Englisch sind |
| Skill-weiter Prompt-Suffix | Ausgabeformat für alle Agenten in diesem Notebook einheitlich |
| Provider-Override | Unterschiedliche Einstellungen für OpenAI vs. Anthropic |

Das Profil wird **einmalig registriert** (`register_harness_profile()`) und wirkt auf alle nachfolgenden `create_deep_agent()`-Aufrufe in dieser Session.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf im Koordinator</font> </br></p>


# Sequenzdiagramm: Ablauf im Koordinator
diagram_seq = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
sequenceDiagram
    autonumber
    actor N as Nutzer
    participant K as Koordinator-Agent
    participant L as LLM (gpt-5.6-sol)
    participant GH as GitHub Repo
    participant D as Dokument-Tools
    participant V as Wissensdatenbank (ChromaDB)
    participant G as analysiere_meeting_kontext
    participant E as aktionspunkte_extrahieren
    participant W as briefing-writer

    Note over K,GH: Setup (einmalig)
    GH-->>K: SKILL.md, WRITER.md, Regelwerke, extract_actions.py

    N->>K: Meeting-Anfrage
    K->>L: Anfrage + Tool-Beschreibungen

    loop Solange das LLM Tools aufruft
        L-->>K: Tool-Call
        Note over K,W: Der Koordinator-Agent führt die Tools aus,<br/>nicht das LLM
        alt Dokumente laden
            K->>D: list_context_documents()
            D-->>K: ["doc1.md", "doc2.txt", ...]
            K->>D: load_context_document(name) [x n]
            D-->>K: Dokumentinhalte
        else Korpus durchsuchen (ergänzend)
            K->>V: suche_wissensdatenbank(frage)
            V-->>K: Treffer mit Quellen
        else Action Items extrahieren
            K->>E: aktionspunkte_extrahieren(kontext)
            E-->>K: Action Items JSON
        else Kontext analysieren
            K->>G: analysiere_meeting_kontext(anfrage, kontext)
            G-->>K: GateOutputResult JSON (Pydantic-validiert)
        else Briefing schreiben
            K->>W: GateOutputResult übergeben
            W-->>K: Formatiertes Briefing
        end
        K->>L: Tool-Ergebnis + bisheriger Kontext
    end

    L-->>K: Finale Antwort
    K-->>N: PMO-Briefing
'''
mermaid(diagram_seq, width=1000)

In [ ]:
# -- VIZ --
from IPython.display import Image, display

display(Image(meeting_agent.get_graph(xray=True).draw_mermaid_png()))

# 6 | Beispiel-Runs

---

| Beispiel | Typ | Szenario |
|----------|-----|----------|
| 1️⃣ | Vorbereitung | Nächster Steuerkreis Projekt Kompass — via Demo-Dokument-Tools |
| 2️⃣ | Nachbereitung | Steuerkreis vom 28.04.2026 — Protokollauszug direkt übergeben |

In [ ]:
import re as _re

def invoke_with_retry(agent, inputs, config, max_retries: int = 5):
    """Wiederholt agent.invoke() bei RateLimitError mit automatischer Wartezeit."""
    for attempt in range(1, max_retries + 1):
        try:
            return agent.invoke(inputs, config=config)
        except Exception as e:
            msg = str(e)
            if "rate_limit_exceeded" not in msg and "429" not in msg:
                raise
            match = _re.search(r"try again in ([\d.]+)s", msg)
            wait = float(match.group(1)) + 2 if match else 15
            print(f"Rate limit (Versuch {attempt}/{max_retries}) — warte {wait:.0f}s …")
            time.sleep(wait)
    raise RuntimeError(f"Rate limit nach {max_retries} Versuchen nicht behoben.")

In [ ]:
with tracing_context(enabled=True):
    # 6.1 Beispiel: Vorbereitung des nächsten Steuerkreises
    aufgabe_1 = dedent("""
        Bereite das Meeting vor und wende den technischen Skill `pmo-briefing` korrekt an.
        Nutze alle verfuegbaren Demo-Dokumente (typ='vorbereitung'),
        extrahiere Action Items mit dem Tool und gib am Ende nur das fertige Briefing aus.
    """).strip()

    print(aufgabe_1)
    print()

    t0 = time.perf_counter()
    result_1 = invoke_with_retry(
        meeting_agent,
        {"messages": [HumanMessage(content=aufgabe_1)]},
        config={
            "recursion_limit": 120,
            "configurable": {"thread_id": "vorbereitung"},
            "run_name": "kap6_briefing_vorbereitung",
            "tags": ["deepagents", "skill", "vorbereitung"],
        },
    )
    latenz_1 = round((time.perf_counter() - t0) * 1000)

    mprint("\n".join([
        "## Laufmetriken Beispiel 1",
        "",
        "| Metrik | Wert |",
        "|--------|------|",
        f"| Nachrichten gesamt | `{len(result_1['messages'])}` |",
        f"| Latenz             | `{latenz_1} ms` |",
        f"| Letzte Rolle       | `{result_1['messages'][-1].type}` |",
    ]))

In [ ]:
briefing_1 = result_1["messages"][-1].content
mprint(briefing_1)

In [ ]:
with tracing_context(enabled=True):
    # 6.2 Beispiel: Steuerkreis-Nachbereitung
    kontext_nach = "\n\n".join(
        [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_NACH.items()]
    )

    aufgabe_2 = dedent(f"""
        Erstelle die Nachbereitung für den folgenden Steuerkreis.
        Wende den technischen Skill `pmo-briefing` korrekt an und extrahiere
        Action Items mit dem Tool. Gib am Ende nur das fertige Briefing aus.

        Meeting-Anfrage:
        - Thema: Steuerkreis Projekt Kompass: Risikoliste-Update
        - Datum: 2026-04-28
        - Teilnehmer: Mara Vogt, Dr. Thomas Brandt, Jonas Reiter
        - Typ: Nachbereitung

        Kontextdokumente:
        {kontext_nach}
    """).strip()

    t0 = time.perf_counter()
    result_2 = invoke_with_retry(
        meeting_agent,
        {"messages": [HumanMessage(content=aufgabe_2)]},
        config={
            "recursion_limit": 120,
            "configurable": {"thread_id": "nachbereitung"},
            "run_name": "kap6_briefing_nachbereitung",
            "tags": ["deepagents", "skill", "nachbereitung"],
        },
    )
    latenz_2 = round((time.perf_counter() - t0) * 1000)

    mprint("\n".join([
        "## Laufmetriken Beispiel 2",
        "",
        "| Metrik | Wert |",
        "|--------|------|",
        f"| Nachrichten gesamt | `{len(result_2['messages'])}` |",
        f"| Latenz             | `{latenz_2} ms` |",
        f"| Letzte Rolle       | `{result_2['messages'][-1].type}` |",
    ]))

    briefing_2 = result_2["messages"][-1].content
    mprint(briefing_2)

In [ ]:
#@markdown   <p><font size="4" color="green">Trace-Analyse</font></p>

time.sleep(2)
show_trace("pmo-briefing", limit=3, show_steps=True)

# 7 | Einordnung

---

<font color="black" size="5">Was an diesem Beispiel wichtig ist</font>

**Skill-Dateien aus dem Repo — nicht im Notebook**

Prompts, Regeln und Skripte kommen aus `github.com/ralf-42/Agenten`.
Das Notebook ist der Ausführungskontext, nicht die Wissensquelle.
Ändert sich `SKILL.md` im Repo, übernimmt das Notebook die neue Version
beim nächsten Run automatisch — ohne manuelles Copy-Paste.

**Structured Output schützt den Writer**

`with_structured_output(GateOutputResult)` stellt sicher, dass der Writer-Sub-Agent
immer vollständige, typisierte Eingaben erhält — unabhängig davon, wie der
Koordinator die Analyse formuliert.

**Grenzen des Musters**

| Grenze | Erklärung |
|--------|-----------|
| Sub-Agent erbt Modell | Writer nutzt `gpt-5.6-sol` statt idealerweise `gpt-5.4-mini` (DeepAgents-Limit) |
| Kein HITL | Für Produktion: `interrupt()` vor Briefing-Versand ergänzen |
| Kein Judge | Skill-Compliance nicht automatisch geprüft |
| Kein Auth | GitHub-Zugriff anonym — für private Repos: Token-Authentifizierung ergänzen |
| Keine autonome PMO-Steuerung | Status und Empfehlungen bleiben quellengebunden; verbindliche Entscheidungen und Aktionen benötigen menschliche Freigabe |

# 8 | Skills für Arbeitsstandards
---


Ein verbreitetes Missverständnis: Skills müssen keine Algorithmen enthalten. Sie können **Stil, Regeln und Qualitätsvorgaben operationalisieren** — alles, was sonst in langen System-Prompts oder unstrukturierten Dokumenten verstreut ist.

Das PMO-Briefing in diesem Modul ist selbst ein Beispiel dafür: Es enthält keine Berechnungen, sondern **Arbeitsstandards**:
- Feste Abschnitte (Agenda, Vorabinfo, Offene Punkte, Nächste Schritte)
- Pflichtfelder mit `[offen]`-Markierung statt Weglassen
- Quellenpflicht für jede Aussage
- Ton- und Formatvorgaben über `writer-format.md`

**Weitere Anwendungsfälle für "Standards-Skills":**

| Typ | Was der Skill operationalisiert | Beispiel |
|-----|--------------------------------|---------|
| Brand Skill | Farben, Typografie, Logoregeln | CraftedWell-Stil für PPT/DOCX |
| Schreibstil | Ton, Formulierungsverbote, Zielgruppe | "Immer Du, keine Passivkonstruktionen" |
| Qualitätsprüfung | Checklisten, Mindestanforderungen | Review-Kriterien vor Veröffentlichung |
| Onboarding-Prozess | Reihenfolge, Pflichtschritte, Eskalation | Briefing-Freigabe-Check |
| Genehmigungsworkflow | Schwellenwerte, Eskalationsstufen | Budgetfreigabe über bestimmtem Betrag |

> 💡 **Kerngedanke:** Ein Skill macht implizites Organisationswissen explizit — und damit für einen Agenten nutzbar. Nicht "was kann das Modell", sondern "was weiß unsere Organisation und wie wird es angewendet".


# 9 | Skill-Qualität und Skill-Tests
---


Wie prüft man, ob ein Skill zuverlässig funktioniert? Die Antwort ist analog zu Unit-Tests in der Softwareentwicklung: **kontrollierte Eingaben mit erwartbarem Verhalten**.

<p><font color='black' size="5">
Grundstruktur eines Skill-Tests
</font></p>


```
tests/
├── test_cases.yaml          # Testfälle mit Eingabe + expected_behavior
├── input/                   # Beispieldokumente als Eingabe
│   ├── steuerkreis_vorbereitung.md
│   └── steuerkreis_notizen.txt
└── expected/                # Referenzausgaben (optional, für strukturierte Formate)
    └── briefing_example.md
```

Ein Testfall (YAML-Format):

```yaml
- name: "Steuerkreis-Vorbereitung mit vollständigen Infos"
  skill: pmo-briefing
  input_files:
    - input/steuerkreis_vorbereitung.md
  prompt: "Bereite das PMO-Briefing für den nächsten Steuerkreis vor."
  expected_behavior: |
    - Abschnitt 'Agenda' vorhanden
    - Abschnitt 'Offene Punkte' vorhanden
    - Keine Spekulationen; fehlende Infos als [offen] markiert
    - Maximal 300 Wörter

- name: "Fehlende Teilnehmerinfos"
  skill: pmo-briefing
  input_files:
    - input/steuerkreis_notizen.txt
  prompt: "Erstelle das Briefing für den Steuerkreis."
  expected_behavior: |
    - Teilnehmer-Abschnitt enthält [offen]-Marker
    - Kein Erfinden von Namen oder Rollen
```


<p><font color='black' size="5">
Drei Qualitätsdimensionen
</font></p>

| Dimension | Frage | Test-Ansatz |
|-----------|-------|-------------|
| **Vollständigkeit** | Sind alle Pflichtabschnitte vorhanden? | Regex / Strukturprüfung |
| **Korrektheit** | Werden nur Fakten aus dem Input verwendet? | LLM-as-Judge |
| **Konsistenz** | Liefert der Skill bei gleicher Eingabe ähnliche Outputs? | Mehrfachausführung + Vergleich |


<p><font color='black' size="5">
Wann sind Skill-Tests besonders wichtig?
</font></p>

- Bei Skills mit **Guardrails** (was darf nicht weggelassen werden)
- Bei **mehrstufigen Workflows** mit Pflichtschritten
- Nach jeder Änderung an `SKILL.md` oder `references/`
- Vor dem **Einsetzen in Production**

> ⚠️ **Faustregel:** Wenn ein Skill in einer realen Entscheidung (Compliance, Freigabe, Kundenkommunikation) eingesetzt wird — dann braucht er Tests, bevor er deployed wird.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Skill-Test Qualitäts-Dimensionen</font> </br></p>

diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    IN["Eingabe
(Dokumente + Prompt)"]
    SK["Skill ausführen
(Agent + SKILL.md)"]
    OUT["Output
(Briefing / Ergebnis)"]
    subgraph Checks ["Qualitätsprüfung"]
        C1["Vollständigkeit
Pflichtabschnitte vorhanden?"]
        C2["Korrektheit
Nur Fakten aus Input?"]
        C3["Konsistenz
Wiederholbar bei gleicher Eingabe?"]
    end
    IN --> SK --> OUT --> Checks
    C1 -->|"✅ / ❌"| Report["Test-Report"]
    C2 -->|"✅ / ❌"| Report
    C3 -->|"✅ / ❌"| Report

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class IN input;
    class C1,C2,C3 metric;
    class OUT,Report output;
'''
mermaid(diagram, width=650)


# A | Aufgaben

---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Erstelle ein Briefing für ein eigenes Meeting-Szenario: Lege ein eigenes Demo-Dokument an und lasse den `meeting_agent` das Briefing erstellen. Das Briefing soll alle Pflichtfelder enthalten.

In [ ]:
# Grundlagen: Eigenes PMO-Briefing
# Startpunkt: meeting_agent und invoke_with_retry aus dem Kapitel "Beispiel-Runs"
mein_meeting = {
    'titel': '...',
    'datum': '...',
    'teilnehmer': [],
    'agenda': [],
}
mein_kontext = '...'  # eigenes Demo-Dokument

# aufgabe = f"Bereite dieses Meeting vor: {mein_meeting}. Kontext: {mein_kontext}"
# result = invoke_with_retry(
#     meeting_agent,
#     {"messages": [HumanMessage(content=aufgabe)]},
#     config={"recursion_limit": 120, "run_name": "kapA_eigenes_briefing", "tags": ["aufgabe"]},
# )
# mprint(result["messages"][-1].content)


**Aufbau**

Teste den `no_context`-Sonderfall: Rufe `analysiere_meeting_kontext()` mit einem Dokument auf, das keinen relevanten Inhalt enthält.

In [ ]:
# Aufbau: no_context-Sonderfall testen
# Startpunkt: analysiere_meeting_kontext() aus dem Kapitel "Pydantic-Modelle & Gate-Analyse"
import json

# Dokument ohne relevanten Inhalt:
leeres_dokument = 'Dieses Dokument enthält keine relevanten Informationen.'

# ergebnis = analysiere_meeting_kontext.invoke({
#     'anfrage': 'Strategiemeeting',
#     'kontext': leeres_dokument,
# })
# gate = json.loads(ergebnis)

# Prüfen: status == 'no_context'?
# print(gate['status'])  # erwartet: 'no_context'


**Vertiefung**

Baue ein Konflikt-Szenario: Zwei Dokumente mit widersprüchlichen Aussagen. Ergänze einen LCEL-Judge, der den Widerspruch erkennt und bewertet.

In [ ]:
# Vertiefung: Konflikt-Szenario + LCEL-Judge
# Startpunkt: LLM-as-Judge

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class ConflictResult(BaseModel):
    konflikt_gefunden: bool
    beschreibung: str = Field(description='Was widerspricht sich?')
    schweregrad: str = Field(description='niedrig / mittel / hoch')

# doc_a = 'Steuerkreis 17.03.2026 (D1): ChromaDB wird für den Prototyp evaluiert.'
# doc_b = 'Steuerkreis 14.04.2026 (D2): Verwaltete Cloud-Vektordatenbank für den produktiven Einsatz; ChromaDB verworfen.'

# judge_llm = init_chat_model(JUDGE)  # gpt-5.4 empfohlen
# judge = judge_llm.with_structured_output(ConflictResult)
# urteil = judge.invoke(f'Dokument A: {doc_a}\nDokument B: {doc_b}', config=run_cfg)
# print(urteil)

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Skills](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/skills.html)
- [Einsteiger DeepAgents](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/einsteiger-deepagents.html)
- [PMO-Assistent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/pmo-assistent-praxis.html)
- [Multi-Agent-Systeme](https://ralf-42.github.io/Agenten/06-multi-agent-erweiterungen/multi-agent-systeme.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
